# Credit Card Approval — Exploratory Data Analysis

This notebook explores the applicant dataset used to train the CreditIQ model: distributions, correlations, and approval-rate breakdowns across key applicant segments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (9, 5)

df = pd.read_csv('../dataset/credit_card_approval.csv')
df.shape

## 1. Data overview

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 2. Missing values & duplicates

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
print('Duplicate rows:', df.duplicated().sum())

## 3. Target distribution

In [ ]:
ax = df['Approved'].value_counts().plot(kind='bar', color=['#F26D6D', '#2FD4B8'])
ax.set_xticklabels(['Rejected', 'Approved'], rotation=0)
ax.set_title('Approval Distribution')
plt.show()

## 4. Numeric feature distributions

In [ ]:
numeric_cols = ['Age', 'Income', 'CreditScore', 'LoanAmount', 'Debt', 'BankBalance', 'MonthlyExpenses', 'YearsEmployed']
df[numeric_cols].hist(bins=30, figsize=(14, 10))
plt.tight_layout()
plt.show()

## 5. Correlation heatmap

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[numeric_cols + ['Approved']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

## 6. Income vs. approval

In [ ]:
sns.boxplot(data=df, x='Approved', y='Income')
plt.title('Income by Approval Outcome')
plt.xticks([0, 1], ['Rejected', 'Approved'])
plt.show()

## 7. Credit score vs. approval

In [ ]:
sns.kdeplot(data=df, x='CreditScore', hue='Approved', fill=True, common_norm=False, palette=['#F26D6D', '#2FD4B8'])
plt.title('Credit Score Distribution by Approval Outcome')
plt.show()

## 8. Loan amount vs. approval

In [ ]:
sns.violinplot(data=df, x='Approved', y='LoanAmount')
plt.xticks([0, 1], ['Rejected', 'Approved'])
plt.title('Loan Amount by Approval Outcome')
plt.show()

## 9. Approval rate by gender

In [ ]:
(df.groupby('Gender')['Approved'].mean() * 100).plot(kind='bar', color='#5B7FF0')
plt.ylabel('Approval Rate (%)')
plt.title('Approval Rate by Gender')
plt.xticks(rotation=0)
plt.show()

## 10. Approval rate by employment status

In [ ]:
(df.groupby('EmploymentStatus')['Approved'].mean() * 100).sort_values().plot(kind='barh', color='#D9B36C')
plt.xlabel('Approval Rate (%)')
plt.title('Approval Rate by Employment Status')
plt.show()

## 11. Approval rate by age bracket

In [ ]:
df['AgeBracket'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 65, 100])
(df.groupby('AgeBracket', observed=True)['Approved'].mean() * 100).plot(kind='bar', color='#2FD4B8')
plt.ylabel('Approval Rate (%)')
plt.title('Approval Rate by Age Bracket')
plt.xticks(rotation=45)
plt.show()

## 12. Pairwise relationships (sample)

In [ ]:
sample = df.sample(min(1500, len(df)), random_state=42)
sns.pairplot(sample, vars=['Income', 'CreditScore', 'LoanAmount', 'Debt'], hue='Approved', palette=['#F26D6D', '#2FD4B8'], plot_kws={'alpha': 0.4, 's': 15})
plt.show()

## 13. Feature importance preview (quick Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

tmp = df.copy()
for col in ['Gender', 'Education', 'MaritalStatus', 'EmploymentStatus', 'ResidenceType']:
    tmp[col] = LabelEncoder().fit_transform(tmp[col].astype(str))
tmp = tmp.drop(columns=['AgeBracket']).fillna(tmp.median(numeric_only=True))

X = tmp.drop(columns=['Approved'])
y = tmp['Approved']
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10).plot(kind='barh', color='#D9B36C')
plt.title('Top 10 Feature Importances (exploratory)')
plt.gca().invert_yaxis()
plt.show()

## Summary

- The target classes are reasonably balanced (~53% approved / 47% rejected) thanks to how the synthetic dataset was generated.
- `CreditScore`, `DebtToIncomeRatio` (engineered in `train.py`), `PreviousDefault`, and `Income` show the strongest relationship with approval outcome.
- Full feature engineering and the final model comparison happen in `Training.ipynb` and `train.py`.